# Competition visuals

Cross-model tables and figures for the paper. Everything is read from results
already in the repository; no model is retrained and no inference is rerun.

The three TSFMs use their common **500-scenario** results throughout.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT / "scripts"))

from evaluate_m6_rps import RANK_COLUMNS, build_ground_truth, load_prices
from exploratory_evaluation import ECE_BINS_PRIMARY, expected_calibration_error

FIGURES = HERE / "figures"
TABLES = HERE / "tables"
FIGURES.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)

NAIVE_RPS = 0.16
ROUNDS = list(range(1, 13))
ROUND_COLUMNS = [f"round_{i:02d}_rps" for i in ROUNDS]

FAMILY = {"Chronos-T5 Base": "TSFM", "Financial Chronos": "TSFM", "Chronos-2": "TSFM",
          "Random Forest": "Feature-based baseline",
          "LightGBM": "Feature-based baseline",
          "Softmax Regression": "Feature-based baseline",
          "FinQBoost": "Competition benchmark",
          "Quasi-Average": "Competition benchmark",
          "Naive": "Naive benchmark"}
TSFMS = ["Chronos-T5 Base", "Financial Chronos", "Chronos-2"]
COLOURS = {"Chronos-T5 Base": "#d62728", "Financial Chronos": "#2ca02c",
           "Chronos-2": "#1f77b4", "Quasi-Average": "#7f3fbf"}


def save_table(frame, path, title, band_rows=(), autosize=False, height=None):
    """Render a dataframe as a clean white-background PNG."""
    width = 1.35 + 1.25 * len(frame.columns)
    fig, ax = plt.subplots(figsize=(width, height or 0.42 * len(frame) + 1.4))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    ax.axis("off")
    text = [[f"{v:.6f}" if isinstance(v, float) else str(v) for v in row]
            for row in frame.to_numpy()]
    rendered = ax.table(cellText=text, colLabels=frame.columns,
                        cellLoc="center", loc="center")
    rendered.auto_set_font_size(False)
    rendered.set_fontsize(9)
    rendered.scale(1, 1.4)
    if autosize:                      # size columns to their contents
        rendered.auto_set_column_width(col=list(range(len(frame.columns))))
    for (row, _), cell in rendered.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if row == 0:
            cell.set_facecolor("#ececec")
            cell.set_text_props(weight="bold")
        else:
            cell.set_facecolor("#f5f5f5" if row - 1 in band_rows else "white")
    ax.set_title(title, fontsize=11, pad=14)
    fig.tight_layout()
    fig.savefig(path, dpi=200, facecolor="white")
    plt.show()

## Loading the per-round RPS for each model

Result versions: TSFMs at 500 scenarios (Chronos-2 uses independent-percentile
sampling), Random Forest and LightGBM no-KNN, Softmax the held-out KNN run,
FinQBoost the 2007-start reimplementation, Quasi-Average the DRE-as-Stock
reimplementation.

In [ ]:
horizon = pd.read_csv(ROOT / "Horizon Experiment" / "results" / "horizon_rps_by_round.csv")
horizon = horizon[(horizon["horizon_days"] == 20) & (horizon["n_scenarios"] == 500)]
TSFM_KEYS = {"Chronos-T5 Base": "Chronos-T5 Base 200M",
             "Financial Chronos": "Financial Chronos 46M",
             "Chronos-2": "Chronos-2 120M (independent sampling)"}

baseline = pd.read_csv(ROOT / "Feature Baselines" / "results" / "feature_baseline_m6_rps.csv")
softmax = pd.read_csv(ROOT / "Feature Baselines" / "results" / "softmax_m6_rps.csv")
finqboost = pd.read_csv(ROOT / "FinQ Boost Reimplementation" / "finqboost_m6_rps.csv")
quasi = pd.read_csv(ROOT / "Quasi-Average Reimplementation DRE-as-Stock" /
                    "results" / "quasi_average_m6_rps.csv")

def from_row(frame, mask=None):
    row = frame if mask is None else frame[mask]
    return row[ROUND_COLUMNS].to_numpy(dtype=float).ravel()

per_round = {}
for label, key in TSFM_KEYS.items():
    block = horizon[horizon["model"] == key].sort_values("round")
    per_round[label] = block["mean_rps"].to_numpy(dtype=float)
per_round["Random Forest"] = from_row(baseline, (baseline["model"] == "rf")
                                      & (baseline["variant"] == "no_knn"))
per_round["LightGBM"] = from_row(baseline, (baseline["model"] == "lgbm")
                                 & (baseline["variant"] == "no_knn"))
per_round["Softmax Regression"] = from_row(softmax)
per_round["FinQBoost"] = from_row(finqboost)
per_round["Quasi-Average"] = from_row(quasi)
per_round["Naive"] = np.full(12, NAIVE_RPS)

for label, values in per_round.items():
    assert values.shape == (12,) and np.isfinite(values).all(), label

# the saved files' own counts, as a cross-check on "rounds beating naive"
print("FinQBoost saved:", int(finqboost["rounds_beating_naive"].iloc[0]),
      "| Quasi-Average saved:", int(quasi["rounds_beating_naive"].iloc[0]))

## 1. Headline model comparison

In [ ]:
headline = pd.DataFrame([{
    "Model": label,
    "Family": FAMILY[label],
    "Overall RPS": float(values.mean()),
    "Rounds beating naive": int((values < NAIVE_RPS).sum()),
} for label, values in per_round.items()]).sort_values("Overall RPS").reset_index(drop=True)

assert headline.loc[headline["Model"] == "FinQBoost", "Rounds beating naive"].iloc[0] == 12
assert headline.loc[headline["Model"] == "Quasi-Average", "Rounds beating naive"].iloc[0] == 9

display = headline.copy()
display["Overall RPS"] = display["Overall RPS"].map("{:.6f}".format)
display["Rounds beating naive"] = display["Rounds beating naive"].map("{}/12".format)
headline = headline.assign(**{"Rounds beating naive":
                              display["Rounds beating naive"]})
headline.to_csv(TABLES / "headline_model_comparison.csv", index=False)
save_table(display, TABLES / "headline_model_comparison.png",
           "Headline model comparison (official M6 evaluation)")
headline

## 2. Per-round RPS (supplementary)

The same models and result versions, round by round. The three TSFMs are banded
at the top.

In [ ]:
order = TSFMS + [m for m in headline["Model"] if m not in TSFMS]
rows = []
for label in order:
    values = per_round[label]
    rows.append({"Model": label,
                 **{f"R{i}": values[i - 1] for i in ROUNDS},
                 "Overall RPS": float(values.mean())})
per_round_table = pd.DataFrame(rows)
per_round_table.to_csv(TABLES / "per_round_rps.csv", index=False)

display = per_round_table.copy()
for column in display.columns[1:]:
    display[column] = display[column].map("{:.4f}".format)
save_table(display, TABLES / "per_round_rps.png",
           "Per-round M6 RPS by model", band_rows=range(len(TSFMS)))
per_round_table.round(4)

## 3. Per-round RPS: TSFMs and the Quasi-Average benchmark

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for label in TSFMS:
    axes[0].plot(ROUNDS, per_round[label], marker="o", color=COLOURS[label], label=label)
axes[1].plot(ROUNDS, per_round["Quasi-Average"], marker="o",
             color=COLOURS["Quasi-Average"], label="Quasi-Average")
for ax, title in zip(axes, ["Time-series foundation models", "Quasi-Average"]):
    ax.axhline(NAIVE_RPS, color="grey", linestyle="--", linewidth=1, label="Naive 0.160")
    ax.set_xticks(ROUNDS)
    ax.set_xlabel("Round")
    ax.set_title(title)
    ax.legend(fontsize=8)
axes[0].set_ylabel("RPS")
fig.tight_layout()
fig.savefig(FIGURES / "per_round_rps_panels.png", dpi=200)
plt.show()

## 4. Confidence vs accuracy at the official 20-day horizon

Confidence is the largest predicted quintile probability and accuracy is whether
that argmax quintile was the realised one, both averaged over the 1,200
asset-rounds. To keep the models comparable, all values are computed here from
each model's saved probability forecasts with the same functions and the same
ground truth. The cell first reproduces the published figures for the three
feature-based models exactly, which confirms the definitions match.

Naive is excluded: a flat [0.2, 0.2, 0.2, 0.2, 0.2] forecast has no unique argmax.

In [ ]:
truth = build_ground_truth(load_prices(), pd.read_csv(ROOT / "Data" / "metadata" /
                                                      "m6_round_schedule.csv"))
truth["true_class"] = truth[RANK_COLUMNS].to_numpy().argmax(axis=1)
key = truth[["round", "symbol", "true_class"]]

def calibration(frame):
    merged = key.merge(frame[["round", "symbol"] + RANK_COLUMNS],
                       on=["round", "symbol"], how="left")
    assert len(merged) == 1200 and not merged[RANK_COLUMNS].isna().any().any()
    probabilities = merged[RANK_COLUMNS].to_numpy(dtype=float)
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
    true_class = merged["true_class"].to_numpy()
    return {"Mean confidence": float(probabilities.max(axis=1).mean()),
            "Accuracy": float((probabilities.argmax(axis=1) == true_class).mean()),
            "ECE": expected_calibration_error(probabilities, true_class, ECE_BINS_PRIMARY)}

detail = pd.read_csv(ROOT / "Horizon Experiment" / "results" / "horizon_forecast_detail.csv")
detail = detail[detail["horizon_days"] == 20]
feature = pd.read_csv(ROOT / "Feature Baselines" / "results" / "feature_baseline_m6_predictions.csv")
softmax_pred = pd.read_csv(ROOT / "Feature Baselines" / "results" / "softmax_m6_predictions.csv")
finq_pred = pd.read_csv(ROOT / "FinQ Boost Reimplementation" /
                        "finqboost_m6_predictions.csv")
quasi_pred = pd.read_csv(ROOT / "Quasi-Average Reimplementation DRE-as-Stock" /
                         "results" / "quasi_average_m6_predictions.csv")

sources = {label: detail[detail["model"] == k] for label, k in TSFM_KEYS.items()}
sources["Random Forest"] = feature[(feature["model"] == "rf") & (feature["variant"] == "no_knn")]
sources["LightGBM"] = feature[(feature["model"] == "lgbm") & (feature["variant"] == "no_knn")]
sources["Softmax Regression"] = softmax_pred
sources["FinQBoost"] = finq_pred
sources["Quasi-Average"] = quasi_pred

calibration_table = pd.DataFrame([{"Model": label, **calibration(frame)}
                                  for label, frame in sources.items()])

# TSFM values must match the horizon experiment's own h=20 diagnostics
horizon_diag = pd.read_csv(ROOT / "Horizon Experiment" / "results" /
                           "horizon_diagnostics.csv")
horizon_diag = horizon_diag[horizon_diag["horizon_days"] == 20].set_index("model")
for label, k in TSFM_KEYS.items():
    row = calibration_table.set_index("Model").loc[label]
    assert np.isclose(row["Mean confidence"], horizon_diag.loc[k, "mean_confidence"], atol=1e-6)
    assert np.isclose(row["Accuracy"], horizon_diag.loc[k, "accuracy"], atol=1e-6)
print("TSFM definitions verified against the horizon experiment metrics")
calibration_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
limits = [0.18, 0.56]
ax.plot(limits, limits, color="black", linewidth=0.9, label="Accuracy = confidence")
markers = {"TSFM": "o", "Feature-based baseline": "s", "Competition benchmark": "^"}
# hand-placed so the tight cluster of baselines stays legible
labels = {"FinQBoost": (-9, 8, "right"), "Random Forest": (10, -3, "left"),
          "Softmax Regression": (10, 3, "left"), "LightGBM": (-10, -9, "right"),
          "Quasi-Average": (0, -30, "center"), "Chronos-2": (10, -10, "left"),
          "Financial Chronos": (10, 4, "left"), "Chronos-T5 Base": (-10, 9, "right")}

for record in calibration_table.itertuples():
    colour = COLOURS.get(record.Model, "#555555")
    ax.scatter(record._2, record.Accuracy, s=70, marker=markers[FAMILY[record.Model]],
               color=colour, zorder=3)
    dx, dy, ha = labels[record.Model]
    ax.annotate(record.Model, (record._2, record.Accuracy), fontsize=8.5, ha=ha,
                textcoords="offset points", xytext=(dx, dy))

for family, marker in markers.items():
    ax.scatter([], [], marker=marker, color="#555555", s=60, label=family)
ax.set_xlim(*limits)
ax.set_ylim(0.19, 0.30)
ax.set_xlabel("Mean confidence")
ax.set_ylabel("Accuracy")
ax.set_title("Confidence vs accuracy at the 20-day M6 horizon")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "confidence_vs_accuracy.png", dpi=200)
plt.show()

Points below the line are more confident than their accuracy supports.

## 5. Calibration metrics

In [ ]:
calibration_table = calibration_table.sort_values("Mean confidence").reset_index(drop=True)
calibration_table.to_csv(TABLES / "calibration_metrics.csv", index=False)

display = calibration_table.copy()
for column in ["Mean confidence", "Accuracy", "ECE"]:
    display[column] = display[column].map("{:.4f}".format)
save_table(display, TABLES / "calibration_metrics.png",
           "Cross-Model Calibration Summary at the 20-Day Horizon")
calibration_table.round(4)

## 6. Quintile distribution by asset type

The observation motivating the Quasi-Average method's asset-type treatment:
stocks reach the extreme quintiles more often than ETFs, and VXX behaves unlike
either. Quintiles are the method's own cross-sectional quintiles of the 20-day
return over 2022, from the DRE-as-Stock reimplementation (DRE sits among the
Stocks).

In [ ]:
QUASI_DIR = ROOT / "Quasi-Average Reimplementation DRE-as-Stock"
sys.path.insert(0, str(QUASI_DIR))
import quasi_average as qa

panel = pd.read_csv(QUASI_DIR / "data" / "adjusted_close_panel.csv",
                    index_col=0, parse_dates=True)
quintiles = qa.quintile_panel(panel).loc["2022"]
metadata = pd.read_csv(ROOT / "Data" / "metadata" / "m6_asset_metadata.csv")
asset_groups = [
    ("Stocks", list(metadata.loc[metadata["asset_type"] == "Stock", "symbol"])),
    ("ETFs", [s for s in metadata.loc[metadata["asset_type"] == "ETF", "symbol"]
              if s != "VXX"]),
    ("VXX", ["VXX"]),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, (label, members) in zip(axes, asset_groups):
    values = quintiles[members].to_numpy().ravel()
    values = values[np.isfinite(values)]
    shares = [np.mean(values == q) for q in range(1, 6)]
    ax.bar(range(1, 6), shares, width=0.7, color="#4c72b0", edgecolor="white")
    ax.axhline(0.2, color="black", linestyle="--", linewidth=1)
    ax.set_xticks(range(1, 6))
    ax.set_xlabel("Quintile (1 = worst return)")
    ax.set_title(f"{label}  (n={values.size:,})", fontsize=10)
axes[0].set_ylabel("Share of observations")
axes[0].text(0.6, 0.205, "uniform 0.2", fontsize=8, color="black")
fig.tight_layout()
fig.savefig(FIGURES / "quasi_average_quintiles_by_asset_type.png", dpi=200)
plt.show()

## 7. TSFM return diagnostics (supplementary)

RMSE of the predicted cumulative return against the realised cumulative return,
using the project's established definition: each scenario path is cumulated as
`expm1(cumsum(daily log returns))` and the point forecast is the MEAN across the
500 scenarios - the centre of the cumulative distribution, not the cumulation of
the centre. The zero-return benchmark predicts 0 for every asset and is a single
benchmark shared by all three models.

Spearman keeps its established definition from the horizon experiment
(cross-sectional correlation of the MEDIAN predicted return against the realised
return, by round, averaged), and is read unchanged from those results.

h=1 uses 11 rounds (Round 4 excluded); h=5, 10, 15 and 20 use all 12 rounds.

In [ ]:
RETURNS_PANEL = ROOT / "Data" / "processed" / "dataset_a_daily_log_returns.csv"
DIAGNOSTIC_HORIZONS = [1, 5, 10, 15, 20]
EXCLUDED_AT_H1 = 4

returns_panel = pd.read_csv(RETURNS_PANEL, parse_dates=["date"]).set_index("date")
SCENARIO_FILES = {
    "Chronos-T5 Base":
        ("TSFM Inference and Sampling/scenario_outputs/chronos_t5_base/chronos_base_round{:02d}_samples500.npz",
         "forecast_samples"),
    "Financial Chronos":
        ("TSFM Inference and Sampling/scenario_outputs/financial_chronos/financial_chronos_round{:02d}_samples500.npz",
         "forecast_samples"),
    "Chronos-2":
        ("TSFM Inference and Sampling/scenario_outputs/chronos2/chronos_2_round{:02d}_scenarios.npz",
         "scenarios"),
}

def cumulative_forecast_and_truth(template, key):
    # per round: mean predicted cumulative return and the realised one, (100, 20)
    predicted, realised = {}, {}
    for rnd in ROUNDS:
        with np.load(ROOT / template.format(rnd), allow_pickle=False) as data:
            samples = data[key][:, :500, :].astype(float)
            symbols = [str(s) for s in data["asset_symbols"]]
            dates = pd.DatetimeIndex([pd.Timestamp(str(d)) for d in data["forecast_dates"]])
        assert samples.shape == (100, 500, 20)
        predicted[rnd] = np.expm1(np.cumsum(samples, axis=2)).mean(axis=1)
        realised[rnd] = np.expm1(np.cumsum(
            returns_panel.loc[dates, symbols].to_numpy(dtype=float).T, axis=1))
    return predicted, realised

def rmse_by_horizon(predicted, realised):
    values = {}
    for h in DIAGNOSTIC_HORIZONS:
        rounds = [r for r in ROUNDS if not (h == 1 and r == EXCLUDED_AT_H1)]
        errors = np.concatenate([predicted[r][:, h - 1] - realised[r][:, h - 1]
                                 for r in rounds])
        values[h] = float(np.sqrt(np.mean(errors ** 2)))
    return values

rmse = {}
for label, (template, key) in SCENARIO_FILES.items():
    predicted, realised = cumulative_forecast_and_truth(template, key)
    rmse[label] = rmse_by_horizon(predicted, realised)

# the zero-return benchmark: forecast 0, so the error is the realised return itself
zero_rmse = {}
for h in DIAGNOSTIC_HORIZONS:
    rounds = [r for r in ROUNDS if not (h == 1 and r == EXCLUDED_AT_H1)]
    truth = np.concatenate([realised[r][:, h - 1] for r in rounds])
    zero_rmse[h] = float(np.sqrt(np.mean(truth ** 2)))

spearman = pd.read_csv(ROOT / "Horizon Experiment" / "results" / "horizon_diagnostics.csv")
spearman = spearman.set_index(["model", "horizon_days"])["mean_spearman_rho"]

rows, shaded = [], []
for label in ["Chronos-2", "Financial Chronos", "Chronos-T5 Base"]:
    rows.append({"Model / Metric": f"{label} RMSE",
                 **{str(h): f"{rmse[label][h]:.4f}" for h in DIAGNOSTIC_HORIZONS}})
    shaded.append(len(rows))
    rows.append({"Model / Metric": f"{label} Spearman",
                 **{str(h): f"{spearman[(TSFM_KEYS[label], h)]:+.4f}"
                    for h in DIAGNOSTIC_HORIZONS}})
rows.append({"Model / Metric": "Zero-return RMSE",
             **{str(h): f"{zero_rmse[h]:.4f}" for h in DIAGNOSTIC_HORIZONS}})

return_diagnostics = pd.DataFrame(rows)
return_diagnostics.to_csv(TABLES / "tsfm_return_diagnostics.csv", index=False)
save_table(return_diagnostics, TABLES / "tsfm_return_diagnostics.png",
           "Time Series Foundation Models Return Diagnostics by Horizon",
           band_rows=shaded, autosize=True, height=3.4)
return_diagnostics

Across horizons, TSFM RMSE is generally close to or worse than the zero-return
benchmark. A small number of model-horizon combinations marginally outperform
zero - Chronos-2 at h=1, Chronos-T5 Base at h=5, and Financial Chronos at h=15
and h=20 - but the differences are small, indicating limited return-magnitude
forecasting skill overall.

## 8. Sampling sensitivity (supplementary)

Overall M6 RPS at each scenario count actually evaluated for each model, from the
completed sampling experiment. Chronos-T5 Base was never run at 1,000 scenarios,
so it simply has no 1,000 column.

In [ ]:
sampling = pd.read_csv(ROOT / "TSFM Inference and Sampling" / "results" / "all_models_rps_by_sample_count.csv")
SAMPLING_KEYS = {"Financial Chronos": "Financial Chronos 46M",
                 "Chronos-2": "Chronos-2 120M (independent sampling)",
                 "Chronos-T5 Base": "Chronos-T5 Base 200M"}

blocks = {}
for label, key in SAMPLING_KEYS.items():
    block = sampling[sampling["model"] == key].sort_values("n_samples")
    blocks[label] = list(zip(block["n_samples"], block["mean_m6_rps"]))
print({k: [n for n, _ in v] for k, v in blocks.items()})

fig, axes = plt.subplots(3, 1, figsize=(6.2, 4.6))
fig.patch.set_facecolor("white")
for ax, (label, values) in zip(axes, blocks.items()):
    ax.set_facecolor("white")
    ax.axis("off")
    table = ax.table(cellText=[[f"{r:.6f}" for _, r in values]],
                     rowLabels=["RPS"],
                     colLabels=[f"{n:,}" for n, _ in values],
                     cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(0.82, 1.35)
    for (row, _), cell in table.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if row == 0:
            cell.set_facecolor("#ececec")
            cell.set_text_props(weight="bold")
        else:
            cell.set_facecolor("white")
    ax.set_title(f"{label}    (scenarios)", fontsize=10, loc="left", pad=2)
fig.suptitle("Sampling sensitivity: overall M6 RPS by scenario count", fontsize=11)
fig.tight_layout()
fig.savefig(TABLES / "sampling_sensitivity.png", dpi=200, facecolor="white")
plt.show()

pd.concat([pd.DataFrame({"Model": label, "Samples": [n for n, _ in v],
                         "RPS": [r for _, r in v]}) for label, v in blocks.items()]
          ).to_csv(TABLES / "sampling_sensitivity.csv", index=False)